# Day 8 Lab: Design and Simulate a Parallel Training Plan

## Scenario
Your team is responsible for training a transformer that may not fit on one GPU. You will build a simplified systems model, compare parallel strategies, visualize memory and pipeline utilization, and defend a final configuration.
 
**Required software:** Python, NumPy, Pandas, Matplotlib  
**GPU required:** No

> This is a planning simulator, not a replacement for profiling a real framework. State every assumption.

## Learning objectives

By the end of the lab, you should be able to:

1. Estimate parameter, gradient, optimizer-state, activation, and buffer memory.
2. Model simplified data, tensor, pipeline, and ZeRO parallelism.
3. Calculate pipeline-bubble utilization.
4. Enumerate valid configurations satisfying `DP × TP × PP = number of GPUs`.
5. Compare feasible configurations by memory and communication risk.
6. Explain why a mathematically feasible configuration may still be slow.

## Part 0: Imports and helper functions

Run the following cell. Do not change it initially.

In [ ]:
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GB = 1024**3

def bytes_to_gib(x):
    return x / GB

print("Environment ready")

## Part 1: Build a baseline training-memory estimator

We use the following simplified accounting:

- Parameters: `num_parameters × parameter_bytes`
- Gradients: `num_parameters × gradient_bytes`
- Optimizer states: `num_parameters × optimizer_bytes_per_parameter`
- Activations: supplied as an estimate
- Temporary buffers and fragmentation: supplied as a reserve

### Task 1.1
Complete `estimate_baseline_memory`.

In [ ]:
def estimate_baseline_memory(
    num_params,
    parameter_bytes=2,
    gradient_bytes=2,
    optimizer_bytes_per_param=8,
    activation_gib=0.0,
    buffer_gib=0.0,
):
    # TODO: calculate each component in GiB.
    parameter_gib = None
    gradient_gib = None
    optimizer_gib = None
    total_gib = None

    return {
        "parameter_gib": parameter_gib,
        "gradient_gib": gradient_gib,
        "optimizer_gib": optimizer_gib,
        "activation_gib": activation_gib,
        "buffer_gib": buffer_gib,
        "total_gib": total_gib,
    }

### Task 1.2: Test your estimator

Estimate memory for 1B, 7B, 13B, and 30B parameter models with:

- BF16 parameters: 2 bytes/parameter
- BF16 gradients: 2 bytes/parameter
- Two FP32 Adam moments: 8 bytes/parameter
- 12 GiB activations
- 4 GiB buffers

Create a DataFrame and answer:

1. Which models fit on one 80 GiB GPU?
2. Why should a design avoid using exactly 80 GiB?

In [ ]:
model_sizes = {
    "1B": 1_000_000_000,
    "7B": 7_000_000_000,
    "13B": 13_000_000_000,
    "30B": 30_000_000_000,
}

# TODO: create one row per model and display a DataFrame.

## Part 2: Model ZeRO stages

Simplified sharding rules for a data-parallel group of size `dp`:

- Stage 0: parameters, gradients, and optimizer states are replicated.
- Stage 1: optimizer states are divided by `dp`.
- Stage 2: optimizer states and gradients are divided by `dp`.
- Stage 3: optimizer states, gradients, and parameters are divided by `dp`.

Activations and buffers are not automatically divided by ZeRO in this model.

### Task 2.1
Complete the function.

In [ ]:
def estimate_zero_memory(
    num_params,
    dp,
    zero_stage,
    parameter_bytes=2,
    gradient_bytes=2,
    optimizer_bytes_per_param=8,
    activation_gib=0.0,
    buffer_gib=0.0,
):
    if zero_stage not in (0, 1, 2, 3):
        raise ValueError("zero_stage must be 0, 1, 2, or 3")

    # TODO: define parameter_factor, gradient_factor, optimizer_factor.
    parameter_factor = None
    gradient_factor = None
    optimizer_factor = None

    # TODO: calculate memory components and total.
    return {}

### Task 2.2: Worked comparison

For a 7B model, `DP = 8`, 12 GiB activations, and 4 GiB buffers:

1. Calculate per-GPU memory for ZeRO stages 0 through 3.
2. Plot a stacked bar chart of parameters, gradients, optimizer states, activations, and buffers.
3. Identify the first stage that fits below a **72 GiB planning limit**. The 8 GiB gap is safety headroom.

In [ ]:
# TODO: calculate a table for stages 0, 1, 2, and 3.
# TODO: create a stacked bar chart.

## Part 3: Add tensor and pipeline parallelism

For this simplified model:

- Tensor parallelism divides parameter, gradient, and optimizer-state memory by `tp`.
- Pipeline parallelism divides those model states by `pp`.
- Pipeline parallelism divides the supplied activation estimate by `pp`.
- Tensor parallel activation sharding is controlled by `tp_activation_factor` because real behavior is architecture dependent.
- ZeRO operates across the data-parallel dimension.

### Task 3.1
Complete the hybrid estimator.

In [ ]:
def estimate_hybrid_memory(
    num_params,
    dp,
    tp,
    pp,
    zero_stage,
    activation_gib,
    buffer_gib,
    parameter_bytes=2,
    gradient_bytes=2,
    optimizer_bytes_per_param=8,
    tp_activation_factor=1.0,
    checkpoint_factor=1.0,
):
    # Start with model-state factors created by TP and PP.
    model_partition = tp * pp

    # TODO: add ZeRO factors across DP.
    parameter_zero_factor = None
    gradient_zero_factor = None
    optimizer_zero_factor = None

    # TODO: compute all components.
    # activation memory should be affected by pp, tp_activation_factor,
    # and checkpoint_factor.
    return {}

### Task 3.2: Sanity checks

Your function should satisfy these qualitative checks:

1. Increasing classic DP alone does not reduce per-GPU memory.
2. Increasing TP or PP reduces per-GPU model-state memory.
3. ZeRO-3 reduces parameters, gradients, and optimizer states across DP.
4. A checkpoint factor below 1 reduces activation memory.

Write at least four `assert` statements.

In [ ]:
# TODO: write sanity-check assertions.

## Part 4: Pipeline utilization

For a simple pipeline with `p` stages and `m` microbatches:

\[
	ext{utilization} = 
rac{m}{m+p-1}
\]

\[
	ext{bubble fraction} = 
rac{p-1}{m+p-1}
\]

### Task 4.1
Implement both functions and verify that utilization plus bubble fraction equals 1.

In [ ]:
def pipeline_utilization(pp, microbatches):
    # TODO
    pass

def pipeline_bubble_fraction(pp, microbatches):
    # TODO
    pass

# TODO: assertions for several values.

### Task 4.2: Worked pipeline experiment

For `PP = 4`, calculate utilization for microbatch counts:

`1, 2, 4, 8, 16, 32`

Plot the result and answer:

1. At what microbatch count does utilization first exceed 80%?
2. Why might you not choose the largest possible number of microbatches?

In [ ]:
# TODO: table and line plot.

## Part 5: Enumerate hybrid configurations

### Hardware and model challenge

- Model: 30B parameters
- GPUs: 32
- Memory per GPU: 80 GiB
- Planning limit: 72 GiB
- Unpartitioned activation estimate: 64 GiB
- Buffer reserve: 5 GiB
- Allowed ZeRO stages: 0, 1, 2, 3
- Allowed DP, TP, and PP values: powers of two
- Constraint: `DP × TP × PP = 32`
- Checkpoint factors to test: `1.0`, `0.5`, `0.25`
- Microbatch choices: `4`, `8`, `16`, `32`

### Task 5.1
Write a configuration generator.

In [ ]:
def powers_of_two_up_to(n):
    values = []
    x = 1
    while x <= n:
        values.append(x)
        x *= 2
    return values


def generate_configs(total_gpus):
    # TODO: yield dictionaries containing dp, tp, and pp.
    # Only include configurations whose product equals total_gpus.
    pass

### Task 5.2: Add a simple risk score

This is not a hardware-performance model. It is a comparison heuristic.

Use:

- TP communication risk: `log2(tp)`
- PP bubble penalty: bubble fraction
- DP communication risk: `log2(dp)` multiplied by 0.5
- ZeRO communication risk: `0.0, 0.25, 0.5, 1.0` for stages 0, 1, 2, 3

Define:

\[
	ext{risk score} = \log_2(TP) + 2	imes	ext{bubble} + 0.5\log_2(DP) + 	ext{ZeRO risk}
\]

Lower is considered better among configurations that fit memory.

In [ ]:
ZERO_RISK = {0: 0.0, 1: 0.25, 2: 0.5, 3: 1.0}

def configuration_risk(dp, tp, pp, zero_stage, microbatches):
    # TODO
    pass

### Task 5.3: Search and rank

Generate every candidate, estimate memory, calculate pipeline utilization and risk, and produce a DataFrame.

Required columns:

- `dp`, `tp`, `pp`, `zero_stage`
- `checkpoint_factor`, `microbatches`
- `memory_gib`, `headroom_gib`
- `pipeline_utilization`
- `risk_score`
- `fits`

Display the ten lowest-risk feasible configurations.

In [ ]:
# TODO: enumerate, calculate, create DataFrame, and rank feasible rows.

## Part 6: Stress-test your chosen plan

Choose one configuration and test it under these changes:

1. Activation estimate increases by 25%.
2. Buffer reserve increases from 5 to 10 GiB.
3. Sequence length doubles. For a first approximation, double activation memory.
4. One pipeline stage is 20% slower than the others.

For the last case, use this simple imbalance approximation:

\[
	ext{balanced throughput factor} = 
rac{	ext{average stage time}}{	ext{maximum stage time}}
\]

### Deliverable
Create a stress-test table and decide whether you would keep or revise your plan.

In [ ]:
# TODO: choose a configuration and build a stress-test DataFrame.

## Part 7: Team design memo

Complete this in the notebook.

### Proposed configuration

- DP:
- TP:
- PP:
- ZeRO stage:
- Checkpoint factor:
- Microbatches:

### Evidence

- Estimated memory per GPU:
- Safety headroom:
- Pipeline utilization:
- Risk score:

### Expected bottleneck

Write 2 to 4 sentences.

### First three measurements on real hardware

1.
2.
3.

### Limitation of this simulator

Write at least two limitations.

## Optional extension A: Visualize the feasible design space

Create a scatter plot:

- x-axis: memory per GPU
- y-axis: risk score
- color: ZeRO stage
- marker shape or annotation: pipeline degree

Mark the 72 GiB planning limit.

In [ ]:
# OPTIONAL TODO

## Optional extension B: Sequence parallelism

Add a `sp` dimension that divides selected activation tensors but not model states. Require `sp` to divide `tp`. Clearly document which activations your simplified model assumes are sharded.

## Submission checklist

- [ ] Completed memory estimator
- [ ] ZeRO comparison table and chart
- [ ] Hybrid estimator with sanity checks
- [ ] Pipeline-utilization plot
- [ ] Ranked configuration table
- [ ] Stress-test table
- [ ] Team design memo
- [ ] Three-minute presentation